[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyTorch and TensorFlow, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)

# Reading Shape Errors &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import contextlib
import os
import re

import numpy as np
import torch
from torch import nn


@contextlib.contextmanager
def c_messages_hidden():
    """Send what C and C++ code writes to the error stream to os.devnull while the block runs."""
    saved = os.dup(2)
    devnull = os.open(os.devnull, os.O_WRONLY)
    os.dup2(devnull, 2)
    try:
        yield
    finally:
        os.dup2(saved, 2)
        os.close(devnull)
        os.close(saved)


os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
with c_messages_hidden():
    import tensorflow as tf
    import keras


def station_network_arrays(n=1200, seed=2025):
    """The made-up stations of the scikit-learn guide, as arrays: four features and two targets."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    shift = np.where(coast_km < 3, 0.8,
            np.where(elevation > 700, -0.5,
            np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), 0.3, 0.0)))
    rng.choice(4, n, p=[0.4, 0.3, 0.1, 0.2])   # each station's owner, which this guide leaves out
    rng.integers(1950, 2025, n)                 # and the year it was installed
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    features = np.column_stack([latitude.round(2), longitude.round(2), elevation.round(0), coast_km.round(1)])
    return features.astype("float32"), jan_mean.round(1).astype("float32"), (may_mean < 2.5).astype("float32")


FEATURES = ["latitude", "longitude", "elevation_m", "coast_km"]
features, jan_mean, snow_on_may_1 = station_network_arrays()
order = np.random.default_rng(0).permutation(len(features))
train_rows, valid_rows, test_rows = order[:800], order[800:1000], order[1000:]
mean, std = features[train_rows].mean(axis=0), features[train_rows].std(axis=0)
scaled = (features - mean) / std

print("torch", torch.__version__.split("+")[0], "| tensorflow", tf.__version__, "| keras", keras.__version__,
      "|", len(features), "stations")

torch 2.11.0 | tensorflow 2.21.0 | keras 3.13.2 | 1200 stations


The worked examples' networks, the five-column table and `shape_error`, rebuilt here so that each
answer runs on its own.

In [2]:
torch.manual_seed(0)
net = nn.Sequential(nn.Linear(4, 32), nn.ReLU(), nn.Linear(32, 1))

keras.utils.set_random_seed(0)
net_keras = keras.Sequential([
    keras.Input((4,), name="features"),
    keras.layers.Dense(32, activation="relu", name="hidden"),
    keras.layers.Dense(1, name="output"),
], name="station_net")

five_columns = np.column_stack([scaled, snow_on_may_1]).astype("float32")


def shape_error(action):
    """Run action, and print the error it raises with the line of its message that gives the sizes."""
    try:
        action()
    except Exception as error:
        text = re.sub(r"\x1b\[[0-9;]*m", "", str(error))
        line = next(line for line in text.splitlines() if "shape" in line or "size" in line)
        print(f"{type(error).__name__}: {line}")
    else:
        print("no error")

**1.** Use `shape_error` to give `net` the first 10 test stations of `five_columns`, and write down what each of the four numbers in the message is.

In [3]:
shape_error(lambda: net(torch.from_numpy(five_columns[test_rows[:10]])))

RuntimeError: mat1 and mat2 shapes cannot be multiplied (10x5 and 4x32)


10 is the batch, the ten test stations, 5 the columns each one has, 4 the features the first layer takes, and 32 that layer's outputs.

**2.** This model raises a shape error: `nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 8), nn.ReLU(), nn.Linear(4, 1))`. Print the shape after every layer for 8 stations to find the layer at fault, then fix it and print the output's shape.

In [4]:
torch.manual_seed(0)
faulty = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 8), nn.ReLU(), nn.Linear(4, 1))
x = torch.from_numpy(scaled[train_rows[:8]])
for position, layer in enumerate(faulty):
    if isinstance(layer, nn.Linear) and x.shape[-1] != layer.in_features:
        print(f"layer {position} expects {layer.in_features} features and is given {x.shape[-1]}")
        break
    x = layer(x)
    print(position, type(layer).__name__, tuple(x.shape))

fixed = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 8), nn.ReLU(), nn.Linear(8, 1))
print("fixed output", tuple(fixed(torch.from_numpy(scaled[train_rows[:8]])).shape))

0 Linear (8, 16)
1 ReLU (8, 16)
2 Linear (8, 8)
3 ReLU (8, 8)
layer 4 expects 4 features and is given 8
fixed output (8, 1)


The loop stops at layer 4, which expects 4 features where the layer before it gives 8. With `nn.Linear(8, 1)` in its place, the model gives one output for each of the 8 stations.

**3.** Build a Keras model for `five_columns`, with `keras.Input((5,))`, a hidden `Dense` layer of 16 and one output, and print the shape of its predictions for the first three test stations.

In [5]:
keras.utils.set_random_seed(0)
five_net = keras.Sequential([
    keras.Input((5,), name="features"),
    keras.layers.Dense(16, activation="relu", name="hidden"),
    keras.layers.Dense(1, name="output"),
], name="five_column_net")
print(five_net.predict(five_columns[test_rows[:3]], verbose=0).shape)

(3, 1)


Three stations go in, and the model gives one prediction for each.

**4.** Predict the January mean for the first test station alone, with `net` and with `net_keras`, giving each the station as a batch of one with `reshape(1, -1)`, and print the shape of each answer.

In [6]:
one_station = scaled[test_rows[0]].reshape(1, -1)
with torch.no_grad():
    print("PyTorch:", tuple(net(torch.from_numpy(one_station)).shape))
print("Keras:  ", net_keras.predict(one_station, verbose=0).shape)

PyTorch: (1, 1)
Keras:   (1, 1)


As a batch of one, the station gets an answer of shape (1, 1) from both frameworks, one station and one output.

**5.** Give `nn.Linear(4, 1)` a tensor of shape (2, 3, 4) through `shape_error`, then print the shape of what it returns. Which dimensions does the layer treat as the batch?

In [7]:
layer = nn.Linear(4, 1)
shape_error(lambda: layer(torch.ones(2, 3, 4)))
print(tuple(layer(torch.ones(2, 3, 4)).shape))

no error
(2, 3, 1)


The call raises no error. `nn.Linear` reads only the last dimension as the features, and treats every dimension before it as part of the batch, here 2 by 3, so the output is (2, 3, 1).

---

&#8592; **Back to:** [Reading Shape Errors](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/07-reading-shape-errors.ipynb)  &nbsp;&middot;&nbsp;  [PyTorch and TensorFlow, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)
